# OpenScience Project: Equatorial Pacific SST Analysis and its Influence in Peruvian Coast.

High resolution MUR SST dataset at AWS available at: https://registry.opendata.aws/mur/

In [25]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import os
import coiled

warnings.simplefilter('ignore') # filter some warning messages
xr.set_options(display_style="html")  #display dataset nicely 

## 1. Cluster: Dask

In [26]:
%%time

cluster_type='Coiled'

if cluster_type == 'Coiled': 
    cluster=coiled.Cluster(
        region='us-west-2',
        arm=True,
        worker_vm_types=['t4g.large'],
        worker_options={"nthreads": 2},
        n_workers= 40,
        name='Vsazonova-us-west-2-large',
        wait_for_workers=False,
        compute_purchase_option='spot_with_fallback',
        software='protocoast-notebook-arm',
        workspace='esip-lab',
    )

Output()

CPU times: user 6.15 s, sys: 448 ms, total: 6.6 s
Wall time: 2min 1s


In [27]:
client = cluster.get_client()
client

<Client: 'tls://10.11.2.202:8786' processes=35 threads=70, memory=250.54 GiB>

## 2. Equatorial Pacific SST Analysis

### Initializing the data

In [28]:
%%time

ds_sst = xr.open_zarr('https://mur-sst.s3.us-west-2.amazonaws.com/zarr-v1',consolidated=True)
ds_sst

CPU times: user 1.92 s, sys: 318 ms, total: 2.24 s
Wall time: 28.2 s


<xarray.Dataset> Size: 117TB
Dimensions:           (time: 6443, lat: 17999, lon: 36000)
Coordinates:
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
  * time              (time) datetime64[ns] 52kB 2002-06-01T09:00:00 ... 2020...
Data variables:
    analysed_sst      (time, lat, lon) float64 33TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
    analysis_error    (time, lat, lon) float64 33TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
    mask              (time, lat, lon) float32 17TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
    sea_ice_fraction  (time, lat, lon) float64 33TB dask.array<chunksize=(5, 1799, 3600), meta=np.ndarray>
Attributes: (12/47)
    Conventions:                CF-1.7
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    MUR = "Multi-scale Ultra-high Resolution"
    creator_email:              ghrsst@podaac.jpl.nasa.gov
    ...                         ...
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    time_coverage_end:          20200116T210000Z
    time_coverage_start:        20200115T210000Z
    title:                      Daily MUR SST, Final product
    uuid:                       27665bc0-d5fc-11e1-9b23-0800200c9a66
    westernmost_longitude:      -180.0

In [29]:
sst = ds_sst['analysed_sst']

cond = (ds_sst.mask==1) & ((ds_sst.sea_ice_fraction<.15) | np.isnan(ds_sst.sea_ice_fraction))
sst_masked = ds_sst['analysed_sst'].where(cond)
sst_masked

<xarray.DataArray 'analysed_sst' (time: 6443, lat: 17999, lon: 36000)> Size: 33TB
dask.array<where, shape=(6443, 17999, 36000), dtype=float64, chunksize=(5, 1799, 3600), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.97 89.98 89.99
  * lon      (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * time     (time) datetime64[ns] 52kB 2002-06-01T09:00:00 ... 2020-01-20T09...
Attributes:
    comment:        "Final" version using Multi-Resolution Variational Analys...
    long_name:      analysed sea surface temperature
    standard_name:  sea_surface_foundation_temperature
    units:          kelvin
    valid_max:      32767
    valid_min:      -32767

In [30]:
sst_elnino = sst_masked.sel(lon=slice(-180,-70), lat=slice(-25,25)) - 273.15 # Slicing EP region for memory safety  

### Diagnostics: monthly mean, anomalies 

In [31]:
%%time
#create a daily climatology and anomaly
climatology_mean = sst_elnino.groupby('time.dayofyear').mean('time',keep_attrs=True,skipna=False)
sst_anomaly = sst_elnino.groupby('time.dayofyear')-climatology_mean  #take out annual mean to remove trends

#create a monthly dataset, climatology, and anomaly
sst_monthly = sst_elnino.resample(time='1MS').mean('time',keep_attrs=True,skipna=False)
climatology_mean_monthly = sst_monthly.groupby('time.month').mean('time',keep_attrs=True,skipna=False)
sst_anomaly_monthly = sst_monthly.groupby('time.month')-climatology_mean_monthly  #take out annual mean to remove trends

sst_anomaly

CPU times: user 8.42 s, sys: 754 ms, total: 9.17 s
Wall time: 9.14 s


<xarray.DataArray 'analysed_sst' (time: 6443, lat: 5001, lon: 11000)> Size: 3TB
dask.array<sub, shape=(6443, 5001, 11000), dtype=float64, chunksize=(2, 1799, 3600), chunktype=numpy.ndarray>
Coordinates:
  * lat        (lat) float32 20kB -25.0 -24.99 -24.98 ... 24.98 24.99 25.0
  * lon        (lon) float32 44kB -180.0 -180.0 -180.0 ... -70.02 -70.01 -70.0
  * time       (time) datetime64[ns] 52kB 2002-06-01T09:00:00 ... 2020-01-20T...
    dayofyear  (time) int64 52kB 152 153 154 155 156 157 ... 15 16 17 18 19 20

### Visualization

In [32]:
import hvplot.xarray
import holoviews as hv
from holoviews.operation.datashader import regrid
hv.extension('bokeh')

### Plot the daily, monthly SST timeseries. SST daily and monthly anomalies. 

we are making a time-series for the Peruvian coast, where we can assess the actual influence of the El Niño event on an example of CHL   

In [33]:
%%time

daily = sst_elnino.sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()
monthly = sst_monthly.sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()

CPU times: user 632 ms, sys: 90.2 ms, total: 722 ms
Wall time: 1min 1s


In [34]:
daily.hvplot(grid=True, xlabel='Time', ylabel='Departure from the mean', title='Daily, Monthly SST Time-series, 2014-2020') * monthly.hvplot(grid=True)

:Overlay
   .Curve.I  :Curve   [time]   (analysed_sst)
   .Curve.II :Curve   [time]   (analysed_sst)

In [35]:
%%time

daily1 = sst_anomaly.drop('dayofyear').sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()
monthly1 = sst_anomaly_monthly.drop('month').sel(time=slice('2014','2020')).sel(lon=-80, lat=-10).load()

CPU times: user 6.54 s, sys: 675 ms, total: 7.21 s
Wall time: 5min 15s


In [36]:
daily1.hvplot(grid=True, xlabel='Time', ylabel='Departure from the mean', title='Daily, Monthly SST Anomalies, 2014-2020') * monthly1.hvplot(grid=True)

:Overlay
   .Curve.I  :Curve   [time]   (analysed_sst)
   .Curve.II :Curve   [time]   (analysed_sst)

### Spatial Visualization

In [38]:
import cartopy.crs as ccrs
import geoviews.feature as gf

In [ ]:
subset = sst_elnino.sel(time='2016-01-01T09')

plot = subset.hvplot.quadmesh(
    x='lon', 
    y='lat', 
    rasterize=True,    
    geo=True, 
    cmap='turbo', 
    frame_width=400
)

basemap = gf.land() * gf.coastline() * gf.borders()
plot * basemap

In [17]:
#sst_elnino = sst.sel(lon=slice(-180,-70), lat=slice(-25,25))

In [39]:
%%time

sst_jan2016 = sst_elnino.sel(time=slice('2016-01-01','2016-02-01')).mean(dim='time')
sst_jan2018 = sst_elnino.sel(time=slice('2018-01-01','2018-02-01')).mean(dim='time')

CPU times: user 15.6 ms, sys: 2.08 ms, total: 17.7 ms
Wall time: 16.8 ms


In [40]:
%%time

sst_diff = (sst_jan2016 - sst_jan2018).load()

CPU times: user 1.34 s, sys: 1.15 s, total: 2.49 s
Wall time: 49.2 s


In [41]:
sst_diff.hvplot.quadmesh(x='lon', y='lat', 
                         geo=True,                 
                         rasterize=True, 
                         cmap='rainbow', 
                         tiles='EsriImagery', title='SST Anomalies: Equatorial Pacific')

:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [lon,lat]   (analysed_sst)

In [42]:
%%time

sst_dy = sst_elnino.sel(time='2016-01-01T09').load()

sst_dy.hvplot.quadmesh(x='lon', y='lat', 
                       geo=True,         
                       rasterize=True, 
                       cmap='rainbow', 
                       projection=ccrs.Orthographic(-145, 35),
                       coastline='110m', title = 'SST Distribution 01-01-2016')

CPU times: user 1.28 s, sys: 1.14 s, total: 2.42 s
Wall time: 41.6 s


:DynamicMap   []
   :Overlay
      .Image.I     :Image   [lon,lat]   (analysed_sst)
      .Coastline.I :Feature   [Longitude,Latitude]

## 3. Chlorophyll experiment

In [8]:
import copernicusmarine
import hvplot.xarray

In [9]:
# turn of some annoying warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [10]:
dataset_id = 'c3s_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M'
ds = copernicusmarine.open_dataset(dataset_id)

INFO - 2026-04-23T14:28:42Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  vsazonova


Copernicus Marine password:

  ········


INFO - 2026-04-23T14:28:52Z - Selected dataset version: "202207"
INFO - 2026-04-23T14:28:52Z - Selected dataset part: "default"


In [11]:
ds

<xarray.Dataset> Size: 147GB
Dimensions:    (time: 328, latitude: 4320, longitude: 8640)
Coordinates:
  * latitude   (latitude) float32 17kB -89.98 -89.94 -89.9 ... 89.9 89.94 89.98
  * longitude  (longitude) float32 35kB -180.0 -179.9 -179.9 ... 179.9 180.0
  * time       (time) datetime64[ns] 3kB 1997-09-01 1997-10-01 ... 2024-12-01
Data variables:
    CHL        (time, latitude, longitude) float32 49GB dask.array<chunksize=(61, 256, 6400), meta=np.ndarray>
    CHL_count  (time, latitude, longitude) float32 49GB dask.array<chunksize=(61, 256, 6400), meta=np.ndarray>
    CHL_error  (time, latitude, longitude) float32 49GB dask.array<chunksize=(61, 256, 6400), meta=np.ndarray>
Attributes: (12/39)
    Conventions:               CF-1.7
    Creation_time:             08:52:23 UTC
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    Naming_authority:          CMEMS
    Netcdf_version_id:         V4
    citation:                  The licensees should respect the Copernicus Ma...
    ...                        ...
    stop_date:                 1997-09-30
    stop_time:                 23:59:00 UTC
    summary:                   These data are Level-4 satellite data (Level 4...
    title:                     c3s_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M
    westernmost_longitude:     -180
    copernicusmarine_version:  2.3.0

In [17]:
%%time
da = ds['CHL'].sel(time='2016-01-01 16:00', method='nearest').load()

CPU times: user 26.9 s, sys: 11.6 s, total: 38.4 s
Wall time: 20.7 s


In [18]:
chl_elnino = da.sel(longitude =slice(-180,-70), latitude =slice(-25,25))

In [22]:
chl_elnino.hvplot(x='longitude', y='latitude', rasterize=True, geo=True, cmap='YlGn', clim = (0,15), tiles='OSM', title='Chlorophyll Concentration (mg/m3)')

:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (Chlorophyll-a concentration in seawater (not log-transformed), generated by as a blended combination of OCI, OCI2, OC2 and OCx algorithms, depending on water class memberships)

In [43]:
client.close()
cluster.close()